In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [1]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)

    # Extract JSON from markdown code block if present
    if "```json" in output:
        output = output.split("```json")[1].split("```")[0].strip()
    elif "```" in output:
        output = output.split("```")[1].split("```")[0].strip()

    return json.loads(output)


dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"Generated {len(dataset)} test cases")
print(json.dumps(dataset, indent=2))

NameError: name 'model' is not defined

In [13]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [14]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [15]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case."""
    results = []
    print("\n>>> Running eval — per-prompt scores print below <<<\n")
    for i, test_case in enumerate(dataset, start=1):
        result = run_test_case(test_case)
        results.append(result)
        task = test_case.get("task", str(test_case))
        preview = task if len(task) <= 100 else task[:97] + "..."
        print(f"[{i}/{len(dataset)}] score={result['score']} | {preview}")

    print("\n" + "=" * 64)
    print("PER-PROMPT SCORES (copy this block if you need a summary)")
    print("=" * 64)
    for i, r in enumerate(results, start=1):
        print(f"  Prompt {i:>2}:  score = {r['score']}")
    print("=" * 64 + "\n")
    return results

In [10]:
import json
from pathlib import Path

dataset_path = Path("dataset.json")
if dataset_path.exists():
    with dataset_path.open() as f:
        dataset = json.load(f)
else:
    # So you can run eval even before generate_dataset succeeds
    dataset = [
        {"task": "Write a Python function that returns True if n is even."},
        {"task": 'Write a JSON object with keys "region" and "vpc_id".'},
        {"task": "Write a regex that matches a dotted IPv4 address."},
    ]
    print("Note: dataset.json not found — using inline sample tasks.\n")

results = run_eval(dataset)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset.json'